In [ ]:
import pandas as pd
import numpy as np

import sqlite3

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

DB_FILE = "../../data/database.db"
TABLE_NAME = "records"

In [ ]:
# Sample input from the raw database
columns = ["step","customer","age","gender","zipcodeori","merchant","zipmerchant","category","customer_amount","fraud"]
input_row = pd.DataFrame(columns=columns)

SAMPLE = [0,'C352968107',"'2'","'M'",'28007','M348934600','28007','es_transportation',39.68,0]

input_row.loc[len(input_row)] = SAMPLE

In [ ]:
def get_last_entry(filters):
    conn = sqlite3.connect(DB_FILE)

    # Build WHERE clause dynamically
    where_clause = " AND ".join([f"{col} = ?" for col in filters.keys()]) if filters else "1=1"
    values = tuple(filters.values())

    query = f"""
        SELECT *
        FROM {TABLE_NAME}
        WHERE {where_clause}
        ORDER BY rowid DESC
        LIMIT 1
    """

    df = pd.read_sql_query(query, conn, params=values)
    conn.close()
    return df

### Get last entry of each, to find rebuild features from the processed dataset


In [ ]:
# Builds all the local customer features
cust_row = get_last_entry({"Customer": input_row.loc[0, "customer"]})

cust_row

In [ ]:
# Builds all the local merchant features
merch_row = get_last_entry({"Merchant": input_row.loc[0, "merchant"]})

merch_row

In [ ]:
# Builds features such as when a customer last shopped with a merchant, or how many times they shopped with a merchant
cust_merch_row = get_last_entry({"Customer": input_row.loc[0, "customer"], "Merchant": input_row.loc[0, "merchant"]})

cust_merch_row

In [ ]:
# Builds feastures of how often a customer shops this category
cust_cat_row = get_last_entry({"Customer": input_row.loc[0, "customer"], f"category_{input_row.loc[0, 'category']}": 1})

cust_cat_row

In [ ]:
# Last row for building the global features
last_row = get_last_entry({})

last_row

In [ ]:
# Convert categorical data back into One-Hot Encoded
for col in cust_row.columns:
    if col.startswith("category") or col.startswith("age") or col.startswith("gender"):
        input_row[col] = 0

input_row.loc[0, f"category_{input_row.loc[0, 'category']}"] = 1
input_row.loc[0, f"age_{input_row.loc[0, 'age']}"] = 1
input_row.loc[0, f"gender_{input_row.loc[0, 'gender']}"] = 1

In [ ]:
# Deletes any rows not in the processed data and adds ones that need to be filled
input_row = input_row.reindex(columns=cust_row.columns, fill_value=0)

input_row

### Build Customer-based Features 
These include: 

- "prev_step" - The last time they did a transaction (-1 if no previous transaction exists)
- "time_since_last_transaction" --
- "transaction_count" - Number of transactions

- "amount_sum" - Sum of all previous transactions
- "avg_amount" - Avergae of all past transactions
- "amount_sq_sum" - Sum of squared amounts
- "M2_amount" - Sum of squared deviation from the mean
- "std_amount" - Standard deviation of past transaction amounts
- "amount_zscore" - How many standard deviation this transaction is from past behaviour

- "avg_amount_ratio" - Ratio of how this given transaction compares to customers average
- "log_amount_ratio" - Log transformation to compress extreme values

- "merchant_count" - Number of times a customer has shopped with this merchant
- "category_count" - Number of times a customer has shopped this category
- "prev_merchant_step" - Last time they shopped with this merchant (-1 if they never have)
- "time_since_last_merchant_transaction" --

In [ ]:
input_row["customer_prev_step"] = cust_row["step"]
input_row["customer_time_since_last_transaction"] = input_row["step"] - input_row["customer_prev_step"]

input_row["customer_transaction_count"] = cust_row["customer_transaction_count"] + 1

input_row["customer_time_since_last_transaction"] = input_row["customer_time_since_last_transaction"].fillna(-1)
input_row["customer_prev_step"] = input_row["customer_prev_step"].fillna(-1)

input_row["customer_amount_sum"] = cust_row["customer_amount"] + cust_row["customer_amount"] 
input_row["customer_amount_sum"] = input_row["customer_amount_sum"].fillna(0)

input_row["customer_avg_amount"] = input_row["customer_amount_sum"] / input_row["customer_transaction_count"]
input_row["customer_avg_amount"] = input_row["customer_avg_amount"].fillna(input_row["customer_amount"])

input_row["customer_amount_sq_sum"] = cust_row["customer_amount_sq_sum"] + (cust_row["customer_amount"] ** 2)
input_row["customer_amount_sq_sum"] = input_row["customer_amount_sq_sum"].fillna(0)#

input_row["customer_M2_amount"] = (
input_row["customer_amount_sq_sum"]
- (input_row["customer_amount_sum"] ** 2) / input_row["customer_transaction_count"]
)

input_row["customer_M2_amount"] = input_row["customer_M2_amount"].fillna(0)
input_row["customer_M2_amount"] = input_row["customer_M2_amount"].clip(lower=0)

input_row["customer_std_amount"] = np.sqrt(input_row["customer_M2_amount"] / (cust_row["customer_transaction_count"]).replace(0, np.nan))
input_row["customer_std_amount"] = input_row["customer_std_amount"].fillna(0)

input_row["customer_avg_amount_ratio"] = input_row["customer_amount"] / input_row["customer_avg_amount"]
input_row["customer_log_amount_ratio"] = np.log1p(input_row["customer_avg_amount_ratio"])

input_row["customer_zscore"] = (input_row["customer_amount"] - input_row["customer_avg_amount"]) / input_row["customer_std_amount"]
input_row["customer_zscore"] = input_row["customer_zscore"].fillna(0)
input_row["customer_zscore"] = input_row["customer_zscore"].replace([np.inf, -np.inf], 0)

input_row["customer_merchant_count"] = cust_merch_row["customer_merchant_count"] + 1
input_row["customer_category_count"] = cust_cat_row["customer_category_count"] + 1

input_row["customer_prev_merchant_step"] = cust_merch_row["step"]
input_row["customer_time_since_last_merchant_transaction"] = input_row["step"] - input_row["customer_prev_merchant_step"]

input_row["customer_time_since_last_merchant_transaction"] = input_row["customer_time_since_last_merchant_transaction"].fillna(-1)
input_row["customer_prev_merchant_step"] = input_row["customer_prev_merchant_step"].fillna(-1)

### Build Merchant-based features 
All of these are the same as customer based features execept for:
- "merchant_fraud_count" - How many times has fraud occured for this merchant
- "merchant_fraud_rate" - Rate of fraud up to this transaction

In [ ]:
input_row["merchant_transaction_count"] = merch_row["merchant_transaction_count"] + 1

input_row["merchant_amount_sum"] = merch_row["customer_amount_sum"] + merch_row["customer_amount"]

input_row["merchant_avg_amount"] = input_row["merchant_amount_sum"] / input_row["merchant_transaction_count"]

input_row["merchant_amount_sq_sum"] = merch_row["merchant_amount_sq_sum"] + (merch_row["customer_amount"] ** 2)

input_row["merchant_M2_amount"] = (
    input_row["merchant_amount_sq_sum"]
    - (input_row["merchant_amount_sum"] ** 2) / input_row["merchant_transaction_count"].replace(0, np.nan)
)

input_row["merchant_std_amount"] = np.sqrt(input_row["merchant_M2_amount"] / (merch_row["merchant_transaction_count"]).replace(0, np.nan))

input_row["merchant_avg_amount_ratio"] = input_row["customer_amount"] / input_row["merchant_avg_amount"]
input_row["merchant_log_amount_ratio"] = np.log1p(input_row["merchant_avg_amount_ratio"])

input_row["merchant_amount_zscore"] = (input_row["customer_amount"] - input_row["merchant_avg_amount"]) / input_row["merchant_std_amount"]

input_row["merchant_fraud_count"] = merch_row["merchant_fraud_count"] + merch_row["merchant_fraud_count"]
input_row["merchant_fraud_rate"] = input_row["merchant_fraud_count"] / input_row["merchant_transaction_count"]
input_row["merchant_fraud_rate"] = input_row["merchant_fraud_rate"].fillna(0)

input_row["merchant_prev_step"] = merch_row["step"]
input_row["merchant_time_since_last_transaction"] = input_row["step"] - input_row["merchant_prev_step"]

input_row["merchant_time_since_last_transaction"] = input_row["merchant_time_since_last_transaction"].fillna(-1)
input_row["merchant_prev_step"] = input_row["merchant_prev_step"].fillna(-1)

### Extract Global Features
Features are the same as both customer and merchant just no longer localised features.

Uses median instead of mean just to extreme outliers within the dataset

In [ ]:
input_row["global_transaction_count"] = last_row["global_transaction_count"] + 1

input_row["global_amount_sum"] = last_row["global_amount_sum"] + last_row["customer_amount"]
input_row["global_amount_sum"] = input_row["global_amount_sum"].fillna(0)

input_row["global_avg_amount"] = input_row["global_amount_sum"] / input_row["global_transaction_count"]
input_row["global_avg_amount"] = input_row["global_avg_amount"].fillna(input_row["customer_amount"])

input_row["global_amount_ratio"] = input_row["customer_amount"] / input_row["global_avg_amount"]
input_row["global_log_amount_ratio"] = np.log1p(input_row["global_amount_ratio"])

# cumulative sum of squares excluding current transaction
input_row["global_amount_sq_sum"] = last_row["global_amount_sq_sum"] + (last_row["customer_amount"] ** 2)
input_row["global_amount_sq_sum"] = input_row["global_amount_sq_sum"].fillna(0)

# rolling M2 using only previous transactions
input_row["global_M2_amount"] = (
    input_row["global_amount_sq_sum"]
    - (input_row["global_amount_sum"] ** 2) / input_row["global_transaction_count"].replace(0, np.nan)
)

input_row["global_M2_amount"] = input_row["global_M2_amount"].fillna(0)
input_row["global_M2_amount"] = input_row["global_M2_amount"].clip(lower=0)

input_row["global_std_amount"] = np.sqrt(input_row["global_M2_amount"] / (last_row["global_transaction_count"]).replace(0, np.nan))
input_row["global_std_amount"] = input_row["global_std_amount"].fillna(0)

input_row["global_z_score"] = (input_row["customer_amount"] - input_row["global_avg_amount"]) / input_row["global_std_amount"]
input_row["global_z_score"] = input_row["global_z_score"].fillna(0)
input_row["global_z_score"] = input_row["global_z_score"].replace([np.inf, -np.inf], 0)

input_row["global_median_amount"] = last_row["global_median_amount"]
input_row["global_median_amount"] = input_row["global_median_amount"].fillna(input_row["customer_amount"])

input_row["global_median_amount_ratio"] = input_row["customer_amount"] / input_row["global_median_amount"]
input_row["global_log_median_amount_ratio"] = np.log1p(input_row["global_median_amount_ratio"])


### Redrops the rows used to build the other features

The rows the models dont expect

In [ ]:
# Time based columns as they are arbitrary as time grows and time_since_last_transactions captures better meaning
input_row.drop(columns=["customer_prev_step", "customer_prev_merchant_step", "merchant_prev_step"], inplace=True)

# Sums and averages are captured in the ratios and z-scores, so can drop them to reduce dimensionality
input_row.drop(columns=["customer_amount_sum", "customer_avg_amount", "merchant_amount_sum", "merchant_avg_amount", "global_amount_sum"], inplace=True)

# Was only used to dervive fraud rate, can drop it now
input_row.drop(columns=["merchant_fraud_count"], inplace=True)

# Dropping from data analyisis, median captures more of the more common transaction amounts, compared to the mean
input_row.drop(columns=["global_avg_amount", "global_amount_ratio", "global_log_amount_ratio", "global_median_amount"], inplace=True)

# From Permutation graphs log values tend to do better
input_row.drop(columns=["customer_avg_amount_ratio", "merchant_avg_amount_ratio", "global_median_amount_ratio"], inplace=True)

# These columns are only used to build std during inference
input_row.drop(columns=["customer_amount_sq_sum", "customer_M2_amount", "merchant_amount_sq_sum","merchant_M2_amount", "global_amount_sq_sum", "global_M2_amount"], inplace=True)

# Index isnt perserved in the DB
input_row.drop(columns=["global_transaction_count"], inplace=True)